# clarai_machine_health_fan: Model Issue Generation & Reviewer Feedback

This notebook populates the **`machine_health.dev.alert_candidates`** unified table for ClairAI Model Observability:
1. Supports **`daily_incremental`** execution mode for scheduled daily production runs:
   - Emits today's batch of candidate issues tagged with current `model_version` (`fan_v1` or `fan_v2`)
   - Simulates human reviewer aging (48-hour SLA review buffer): transitions issues older than 48 hours from `New` to `Open` (Accepted) or `Removed` (Rejected)
2. Supports **`seed_baseline`** execution mode for one-time initialization or scenario reset:
   - Promoted Baseline: `fan_v1` at **81.0%** acceptance (210 issues: 170 accepted, 40 rejected)
   - Accidental Deployment: `fan_v2` drops to **45.7%** (35.3 point dip) with **ModelOversensitive** surge (11 of 25)
   - Pending review: 91 issues in `New` status (< 48 hours old)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta
import random

# ============================================================
# WIDGET DECLARATIONS
# ============================================================
dbutils.widgets.dropdown("execution_mode", "daily_incremental", ["daily_incremental", "seed_baseline"])
dbutils.widgets.text("model_version", "fan_v1")
dbutils.widgets.text("scenario_stage", "accidental_deployment")
dbutils.widgets.dropdown("time_anchor", "relative_to_now", ["relative_to_now", "scenario"])
dbutils.widgets.dropdown("clean_table", "false", ["true", "false"])
dbutils.widgets.text("job_id", "745174521807184")
dbutils.widgets.text("job_run_id", "0")
dbutils.widgets.text("workspace_id", "7474657594011271")

exec_mode = dbutils.widgets.get("execution_mode").strip().lower()
model_ver = dbutils.widgets.get("model_version").strip().lower() or "fan_v1"
stage = dbutils.widgets.get("scenario_stage").strip().lower()
time_anchor = dbutils.widgets.get("time_anchor").strip().lower()
clean_table = dbutils.widgets.get("clean_table").strip().lower() == "true"

JOB_ID = dbutils.widgets.get("job_id").strip() or "745174521807184"
JOB_RUN_ID = dbutils.widgets.get("job_run_id").strip() or "0"
JOB_NAME = "clarai_machine_health_fan"

CAT = "machine_health.prod"
FEED_TABLE = f"{CAT}.alert_candidates"

# MLflow Run IDs (Clean 32-character IDs)
MLFLOW_RUN_V101 = "bb1b3a15559f43dd9993b3b28c24282c"
MLFLOW_RUN_V102 = "843a6a49a1b7479cbe3f0e0c33f10b37"

print(f"Mode='{exec_mode}', model_version='{model_ver}', stage='{stage}', time_anchor='{time_anchor}', clean_table={clean_table}")

In [0]:
# Schema definition for alert_candidates
feed_schema = StructType([
    StructField("issue_id", StringType(), False),
    StructField("sensor_id", StringType(), False),
    StructField("machine_type", StringType(), False),
    StructField("reading_time", TimestampType(), False),
    StructField("axis", StringType(), False),
    StructField("is_healthy", LongType(), False),
    StructField("model_tag", StringType(), False),
    StructField("decision_note", StringType(), False),
    StructField("status", StringType(), False),
    StructField("removal_reason", StringType(), True),
    StructField("mlflow_run_id", StringType(), False),
    StructField("job_id", StringType(), False),
    StructField("job_run_id", StringType(), False)
])

feed_records = []

def add_issue(issue_num, sensor_num, m_tag, ml_run, dt, status, reason, j_run):
    issue_id = f"ISS-F-{issue_num:04d}"
    sensor_id = f"FAN-{sensor_num:03d}"
    feed_records.append((
        issue_id,
        sensor_id,
        "Fan",
        dt,
        "X",
        0,
        m_tag,
        "rule_flagged",
        status,
        reason,
        ml_run,
        JOB_ID,
        str(j_run)
    ))

### Execution Branch: Seed Baseline vs Daily Incremental

In [0]:
if exec_mode == "seed_baseline" or clean_table:
    # ============================================================
    # SEED BASELINE MODE (Full 14-day history + candidate deployment)
    # ============================================================
    print("--- Executing SEED BASELINE Mode ---")
    
    if time_anchor == "relative_to_now":
        now = datetime.utcnow().replace(minute=10, second=0, microsecond=0)
        anchor_14_oct = now - timedelta(days=2)
    else:
        anchor_14_oct = datetime(2026, 10, 14, 9, 10, 0)
        
    cur_issue = 1000
    
        # 1. Baseline fan_v1 (14 days, 212 issues, 175 accepted, 37 rejected = 82.5%)
        rejection_days = {
            0: 3, 1: 3, 2: 2, 3: 3, 4: 2, 5: 3, 6: 3, 
            7: 2, 8: 3, 9: 3, 10: 2, 11: 3, 12: 2, 13: 3
        }
        for day in range(14):
            run_date = anchor_14_oct - timedelta(days=(14 - day))
            job_run = 1044810000 + day
            n_rem = rejection_days.get(day, 3)
            day_statuses = [("Removed", "EquipmentIsHealthy")] * (n_rem // 2) + \
                           [("Removed", "DuplicateIssue")] * (n_rem - (n_rem // 2) - 1 if n_rem > 2 else 1) + \
                           ([("Removed", "ModelOversensitive")] if day % 3 == 0 else [("Removed", "SensorGlitch")])
            day_issue_count = 17 if day == 13 else 15
            day_statuses = day_statuses[:n_rem] + [("Open", None)] * (day_issue_count - n_rem)
            
            for i, (st, rsn) in enumerate(day_statuses):
                cur_issue += 1
                sensor_num = 1 if (cur_issue % 2 == 0) else 2
                add_issue(cur_issue, sensor_num, "fan_v1", MLFLOW_RUN_V101, run_date + timedelta(seconds=i*10), st, rsn, job_run)

    # 2. Accidental Deployment: fan_v2 (Day 1, 2, 3)
    if stage in ["all", "accidental_deployment"]:
        # Day 1: 46 issues (21 accepted, 25 rejected = 45.7%, 11 ModelOversensitive)
        day1_date = anchor_14_oct
        job_run_14 = 1044811014
        removals_14 = (
            ["ModelOversensitive"] * 11 +
            ["EquipmentIsHealthy"] * 9 +
            ["DuplicateIssue"] * 3 +
            ["SensorGlitch"] * 2
        )
        statuses_14 = [("Removed", r) for r in removals_14] + [("Open", None)] * 21
        random.seed(42)
        random.shuffle(statuses_14)
        for idx, (st, rsn) in enumerate(statuses_14):
            iss_num = 2001 + idx
            sensor_num = 1 if (idx % 2 == 0) else 2
            add_issue(iss_num, sensor_num, "fan_v2", MLFLOW_RUN_V102, day1_date + timedelta(seconds=idx*5), st, rsn, job_run_14)
            
        # Day 2: 44 issues (18 accepted, 26 rejected = 40.9%)
        day2_date = anchor_14_oct + timedelta(days=1)
        job_run_15 = 1044811015
        removals_15 = (
            ["ModelOversensitive"] * 12 +
            ["EquipmentIsHealthy"] * 9 +
            ["DuplicateIssue"] * 3 +
            ["SensorGlitch"] * 2
        )
        statuses_15 = [("Removed", r) for r in removals_15] + [("Open", None)] * 18
        random.seed(43)
        random.shuffle(statuses_15)
        for idx, (st, rsn) in enumerate(statuses_15):
            iss_num = 2047 + idx
            sensor_num = 1 if (idx % 2 == 0) else 2
            add_issue(iss_num, sensor_num, "fan_v2", MLFLOW_RUN_V102, day2_date + timedelta(seconds=idx*8), st, rsn, job_run_15)
            
        # Day 3: 47 issues (19 accepted, 28 rejected = 40.4%)
        day3_date = anchor_14_oct + timedelta(days=2)
        job_run_16 = 1044811016
        removals_16 = (
            ["ModelOversensitive"] * 13 +
            ["EquipmentIsHealthy"] * 9 +
            ["DuplicateIssue"] * 4 +
            ["SensorGlitch"] * 2
        )
        statuses_16 = [("Removed", r) for r in removals_16] + [("Open", None)] * 19
        random.seed(44)
        random.shuffle(statuses_16)
        for idx, (st, rsn) in enumerate(statuses_16):
            iss_num = 2091 + idx
            sensor_num = 1 if (idx % 2 == 0) else 2
            add_issue(iss_num, sensor_num, "fan_v2", MLFLOW_RUN_V102, day3_date + timedelta(seconds=idx*6), st, rsn, job_run_16)

    # Write Seed records
    df_seed = spark.createDataFrame(feed_records, feed_schema)
    df_seed.write.mode("overwrite").option("replaceWhere", "machine_type = 'Fan'").saveAsTable(FEED_TABLE)
    print(f"Seeded {len(feed_records)} records into {FEED_TABLE}.")

else:
    # ============================================================
    # DAILY INCREMENTAL MODE (Real-time scheduled daily run)
    # ============================================================
    print("--- Executing DAILY INCREMENTAL Mode ---")
    now = datetime.utcnow().replace(second=0, microsecond=0)
    
    # 1. Determine next issue ID sequence
    max_id_row = spark.sql(f"""
        SELECT MAX(CAST(SUBSTRING(issue_id, 7) AS INT)) as max_id 
        FROM {FEED_TABLE} 
        WHERE issue_id LIKE 'ISS-F-%'
    """).first()
    next_issue = (max_id_row.max_id if max_id_row and max_id_row.max_id else 2137) + 1
    
    # 2. Generate Today's Batch of Candidate Issues
    active_mlflow_run = MLFLOW_RUN_V102 if model_ver == "fan_v2" else MLFLOW_RUN_V101
    
    # Volume: fan_v2 produces surge of ~44-48 issues/day; fan_v1 produces normal ~15-17 issues/day
    issue_count = random.randint(44, 48) if model_ver == "fan_v2" else random.randint(15, 17)
    print(f"Generating {issue_count} candidate issues for model_version='{model_ver}' at {now}")
    
    for idx in range(issue_count):
        cur_num = next_issue + idx
        sensor_num = 1 if (idx % 2 == 0) else 2
        issue_time = now - timedelta(minutes=random.randint(5, 55), seconds=random.randint(0, 59))
        add_issue(
            issue_num=cur_num,
            sensor_num=sensor_num,
            m_tag=model_ver,
            ml_run=active_mlflow_run,
            dt=issue_time,
            status="New",
            reason=None,
            j_run=JOB_RUN_ID
        )
        
    df_new = spark.createDataFrame(feed_records, feed_schema)
    df_new.write.mode("append").saveAsTable(FEED_TABLE)
    print(f"Appended {len(feed_records)} new issues for today's run.")
    
    # 3. Simulate Human Reviewer SLA (Review pending issues >= 48 hours old)
    review_cutoff = now - timedelta(hours=48)
    cutoff_str = review_cutoff.strftime("%Y-%m-%d %H:%M:%S")
    
    pending_issues = spark.sql(f"""
        SELECT issue_id, model_tag, reading_time 
        FROM {FEED_TABLE}
        WHERE machine_type = 'Fan' 
          AND status = 'New' 
          AND reading_time <= timestamp('{cutoff_str}')
    """).collect()
    
    print(f"Found {len(pending_issues)} pending issues older than 48 hours ready for review.")
    
    if pending_issues:
        reviewed_updates = []
        random.seed(int(now.timestamp()) % 10000)
        
        for p in pending_issues:
            p_id = p["issue_id"]
            p_tag = p["model_tag"]
            
            if p_tag == "fan_v2":
                # Candidate model (oversensitive): 40-45% accepted, 55-60% rejected (~44% of rejections are ModelOversensitive)
                v2_threshold = random.choice([0.404, 0.409, 0.423, 0.457])
                is_accepted = random.random() < v2_threshold
                if is_accepted:
                    reviewed_updates.append((p_id, "Open", None))
                else:
                    r_val = random.random()
                    if r_val < 0.456:
                        reason = "ModelOversensitive"
                    elif r_val < 0.80:
                        reason = "EquipmentIsHealthy"
                    elif r_val < 0.92:
                        reason = "DuplicateIssue"
                    else:
                        reason = "SensorGlitch"
                    reviewed_updates.append((p_id, "Removed", reason))
            else:
                # Baseline fan_v1: realistic random acceptance between 80% and 85% (80.0%, 82.4%, 84.0%, 85.0%, 86.7%)
                v1_threshold = random.choice([0.800, 0.824, 0.840, 0.850, 0.867])
                is_accepted = random.random() < v1_threshold
                if is_accepted:
                    reviewed_updates.append((p_id, "Open", None))
                else:
                    r_val = random.random()
                    if r_val < 0.50:
                        reason = "EquipmentIsHealthy"
                    elif r_val < 0.80:
                        reason = "DuplicateIssue"
                    else:
                        reason = "SensorGlitch"
                    reviewed_updates.append((p_id, "Removed", reason))
                    
        # Apply updates via Delta MERGE
        review_schema = StructType([
            StructField("issue_id", StringType(), False),
            StructField("new_status", StringType(), False),
            StructField("new_reason", StringType(), True)
        ])
        df_updates = spark.createDataFrame(reviewed_updates, review_schema)
        df_updates.createOrReplaceTempView("review_updates_view")
        
        spark.sql(f"""
            MERGE INTO {FEED_TABLE} t
            USING review_updates_view u
            ON t.issue_id = u.issue_id
            WHEN MATCHED THEN UPDATE SET
                t.status = u.new_status,
                t.removal_reason = u.new_reason
        """)
        print(f"Reviewed and updated {len(reviewed_updates)} matured issues in {FEED_TABLE}.")

### Verification Query: What ClairAI Sees in alert_candidates

In [0]:
%sql
-- Acceptance metrics grouped by model_tag (ClairAI View)
WITH scored_issues AS (
  SELECT 
    model_tag,
    issue_id,
    MAX(status) as status,
    MAX(removal_reason) as removal_reason
  FROM machine_health.prod.alert_candidates
  WHERE machine_type = 'Fan' AND issue_id IS NOT NULL
  GROUP BY model_tag, issue_id
)
SELECT 
  model_tag,
  COUNT(issue_id) as total_distinct_issues,
  SUM(CASE WHEN status IN ('New', 'Open') THEN 1 ELSE 0 END) as accepted_issues,
  SUM(CASE WHEN status = 'Removed' THEN 1 ELSE 0 END) as rejected_issues,
  ROUND(SUM(CASE WHEN status IN ('New', 'Open') THEN 1 ELSE 0 END) * 100.0 / COUNT(issue_id), 1) as acceptance_rate_pct
FROM scored_issues
GROUP BY model_tag
ORDER BY model_tag;

In [0]:
%sql
-- Status Breakdown including pending review (New)
SELECT 
  model_tag,
  status,
  COUNT(DISTINCT issue_id) as count
FROM machine_health.prod.alert_candidates
WHERE machine_type = 'Fan' AND issue_id IS NOT NULL
GROUP BY model_tag, status
ORDER BY model_tag, status;